In [1]:
# Import von Bibliotheken
from datasets import load_dataset
!pip install pypdf
import urllib.request
import io,re
import json
from pypdf import PdfReader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install textstat
import textstat
!pip install simplemma
!pip install lexicalrichness
import simplemma
from simplemma import simple_tokenizer
from lexicalrichness import LexicalRichness
!pip install nltk
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = stopwords.words('german')

[nltk_data] Downloading package punkt to /home/haloh001/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/haloh001/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Wahlprogramme**

In [2]:
# Einlesen der Langwahlprogramme (aus JSON-Format in ein Wörterbuch)

# für CDU/CSU
with open("cdu-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    CDU_text_json = json.load(datei)

# für SPD
with open("spd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    SPD_text_json = json.load(datei)

# für Die Linke
with open("linke-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Linke_text_json = json.load(datei)

# für die AfD
with open("afd-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    AfD_text_json = json.load(datei)

# für Bündnis 90/Grüne
with open("gruene-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    Grüne_text_json = json.load(datei)

# für FDP
with open("fdp-wahlprogramm_2025_absatzweise.json", "r", encoding="utf-8") as datei:
    FDP_text_json = json.load(datei)

In [3]:
# Überführen aller 'eigentlichen' Texte in eine Liste pro Partei, d.h. ohne Kapitelüberschriften

# Suchfunktion in den "Partei-Wörterbüchern"

def finde_gefilterte_texte(struktur):
    ergebnisse = []
    
    # Fall A: Wenn das aktuelle Element ein Dictionary ist
    if isinstance(struktur, dict):
        # Typ/Art auslesen (liefert None, wenn der Schlüssel fehlt)
        aktueller_typ = struktur.get('type') or struktur.get('art')            # nur Suchen in diesen Elementen
        
        # Prüfen, ob 'text' existiert und der gefundene Typ gültig ist
        if 'text' in struktur and aktueller_typ in ['paragraph', 'bullet', 'absatz']:   # nur Texte überführen mit diesem Typ
            ergebnisse.append(struktur['text'])
        
        # Tiefer in alle Werte schauen für eventuelle Verschachtelungen
        for wert in struktur.values():
            ergebnisse.extend(finde_gefilterte_texte(wert))
                
    # Fall B: Wenn das aktuelle Element eine Liste ist
    elif isinstance(struktur, list):
        for element in struktur:
            ergebnisse.extend(finde_gefilterte_texte(element))
            
    return ergebnisse

CDU_text = finde_gefilterte_texte(CDU_text_json)
SPD_text = finde_gefilterte_texte(SPD_text_json)
Linke_text = finde_gefilterte_texte(Linke_text_json)
Grüne_text = finde_gefilterte_texte(Grüne_text_json)
AfD_text = finde_gefilterte_texte(AfD_text_json)
FDP_text = finde_gefilterte_texte(FDP_text_json)

In [4]:
# Überführen aller 'eigentlichen' Texte in einen String pro Partei, d.h. ohne Kapitelüberschriften

# für CDU/CSU
CDU_text_str = [text if text.endswith('.') else text + '.' for text in CDU_text]   # ein "Punkt" wird gesetzt bei Bulletpoints
CDU_text = " ".join(CDU_text_str)

# für SPD
SPD_text_str = [text if text.endswith('.') else text + '.' for text in SPD_text]
SPD_text = " ".join(SPD_text_str)

# für Die Linke
Linke_text_str = [text if text.endswith('.') else text + '.' for text in Linke_text]
Linke_text = " ".join(Linke_text_str)

# für die AfD
AfD_text_str = [text if text.endswith('.') else text + '.' for text in AfD_text]
AfD_text = " ".join(AfD_text_str)

# für Bündnis 90/Grüne
Grüne_text_str = [text if text.endswith('.') else text + '.' for text in Grüne_text]
Grüne_text = " ".join(Grüne_text_str)

# für FDP
FDP_text_str = [text if text.endswith('.') else text + '.' for text in FDP_text]
FDP_text = " ".join(FDP_text_str)

In [5]:
# Bereinigung der Texte - TEIL 1 -> um Sonderzeichen usw., siehe Kommentar neben entsprechendem reg-Ausdruck

def str_bereinigen(i):
    i = re.sub(r'-\s+', '', i)                               # Ersetzt den Bindestrich gefolgt von einem oder mehreren Leerzeichen durch nichts
    i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)            # Sonderzeichen-Filter, löscht alles, was nicht ausdrücklich erhalten bleiben soll
    i = re.sub(r'([A-Za-zÄÖÜäöüß])\1{3,}', r'\1\1\1', i)     # Begrenzung von extremen Wiederholungen
    i = re.sub(r'\s+', ' ', i).strip()                       # Normalisierung von Whitespaces (ersetzt durch ein Leerzeichen)
    return i
    
CDU_text = str_bereinigen(CDU_text)
SPD_text = str_bereinigen(SPD_text)
Grüne_text = str_bereinigen(Grüne_text)
AfD_text = str_bereinigen(AfD_text)
Linke_text = str_bereinigen(Linke_text)
FDP_text = str_bereinigen(FDP_text)


In [6]:
# Bereinigen der zuvor bereinigten Texte - TEIL 2 => Umformen von Abkürzungen in Text um Eindruck von einem Satzende zu vermeiden

def bereinige_abkuerzungen(text):
    # Dictionary mit den Top 20 Ersetzungen (Regex-Muster als Key)
    ersetzungen = {
        r'\bz\.\s*B\.': 'zum Beispiel',   # r'\bz\.\s*B\.\b': 'zum Beispiel',
        r'\bu\.\s*a\.': 'unter anderem',
        r'\bd\.\s*h\.': 'das heißt',
        r'\bbzw\.': 'beziehungsweise',
        r'\bbzw': 'beziehungsweise',   # Variante ohne Punkt
        r'\bsog\.': 'sogenannte',
        r'\bca\.': 'circa',
        r'\bevtl\.': 'eventuell',
        r'\binkl\.': 'inklusive',
        r'\betc\.': 'et cetera',
        r'\bvgl\.': 'vergleiche',
        r'\bs\.': 'siehe',
        r'\bu\.\s*v\.\s*m\.': 'und vieles mehr',
        r'\bu\.\s*ä\.': 'und ähnliche',
        r'\bggf\.': 'gegebenenfalls',
        r'\bzzgl\.': 'zuzüglich',
        r'\bebd\.': 'ebenda',
        r'\bo\.\s*g\.': 'oben genannte',
        r'\bu\.\s*g\.': 'unten genannte',
        r'\bi\.\s*d\.\s*R\.': 'in der Regel',
        r'\bv\.\s*a\.': 'vor allem',

        # 2. Titel & Personen (Verhindern Satzabbruch mitten im Fluss)
        r'\bDr\.': 'Doktor',
        r'\bProf\.': 'Professor',
        r'\bFr\.': 'Frau',
        r'\bHr\.': 'Herr',

        # 3. Wortlaengen- & Silben-Verfaelscher
        r'\bbspw\.': 'beispielsweise',
        r'\bbspw': 'beispielsweise',   # Variante ohne Punkt
        r'\bbsp\.': 'Beispiel',
        r'\bJh\.': 'Jahrhundert',
        r'\bMio\.': 'Millionen',
        r'\bMrd\.': 'Milliarden',
        r'\bbetr\.': 'betreffend',
        r'\bbezgl\.': 'bezüglich',
        r'\bvs\.': 'versus'
    }

    # Text Schritt für Schritt bereinigen
    for muster, ersetzung in ersetzungen.items():
        # flags=re.IGNORECASE sorgt dafür, dass auch "Z.B." oder "Bzw." gefunden werden
        text = re.sub(muster, ersetzung, text, flags=re.IGNORECASE)

    return text

CDU_text = bereinige_abkuerzungen(CDU_text)
SPD_text = bereinige_abkuerzungen(SPD_text)
Grüne_text = bereinige_abkuerzungen(Grüne_text)
AfD_text = bereinige_abkuerzungen(AfD_text)
Linke_text = bereinige_abkuerzungen(Linke_text)
FDP_text = bereinige_abkuerzungen(FDP_text)


**1. Teil: lexical richness (für Wahlprogramme)**

In [7]:
# ohne stopwords

# automatisierte Funktion  QUELLE: Gemini
def clean_and_analyze_german(text, window_size=100):
    # Bindestriche durch Leerzeichen ersetzen, damit Wörter nicht verschmelzen
    text_cleaned = re.sub(r'-', ' ', text)
    text_cleaned = re.sub(r'[^a-zA-ZäöüÄÖÜß\s]', '', text)
    #text_cleaned = text_cleaned.lower()    # inaktiv

    # 2. Tokenisierung
    # und zusätzlich leere Strings rausfiltern, die durch re.sub entstehen könnten
    tokens = re.findall(r'\b\w+\b', text_cleaned)
    tokens = [t for t in tokens if t.strip()]

    # Bereinigen um stopwords
    tokens_clear = [word for word in tokens if word not in stopwords]

    # Lemmatisierung (behält die originale Wortanzahl und Reihenfolge bei)
    lemmatized_tokens = [simplemma.lemmatize(t, lang='de') for t in tokens_clear]
    # erst jetzt umwandeln in Kleinbuchstaben
    lemmatized_tokens = [t.lower() for t in lemmatized_tokens]   # neu gegen inaktiv
    text_string = " ".join(lemmatized_tokens)

    # Prüfen, ob genug Wörter nach der Reinigung übrig sind
    if len(lemmatized_tokens) < window_size:
        return f"Fehler: Text hat nur {len(lemmatized_tokens)} Wörter (Fenster: {window_size})."

    # 3. MATTR Berechnung
    lex = LexicalRichness(text_string)
    mattr_score = lex.mattr(window_size=window_size)

    return mattr_score

# Berechnung
CDU_mattr_WP = clean_and_analyze_german(CDU_text)
SPD_mattr_WP = clean_and_analyze_german(SPD_text)
Linke_mattr_WP = clean_and_analyze_german(Linke_text)
AfD_mattr_WP = clean_and_analyze_german(AfD_text)
Grüne_mattr_WP = clean_and_analyze_german(Grüne_text)
FDP_mattr_WP = clean_and_analyze_german(FDP_text)

In [8]:
# mit allen Wörter, d.h. auch stopwords

# automatisierte Funktion  QUELLE: Gemini
def clean_and_analyze_german(text, window_size=100):
    # Bindestriche durch Leerzeichen ersetzen, damit Wörter nicht verschmelzen
    text_cleaned = re.sub(r'-', ' ', text)
    text_cleaned = re.sub(r'[^a-zA-ZäöüÄÖÜß\s]', '', text)
    #text_cleaned = text_cleaned.lower()    # inaktiv

    # 2. Tokenisierung
    # und zusätzlich leere Strings rausfiltern, die durch re.sub entstehen könnten
    tokens = re.findall(r'\b\w+\b', text_cleaned)
    tokens = [t for t in tokens if t.strip()]

    # Lemmatisierung (behält die originale Wortanzahl und Reihenfolge bei)
    lemmatized_tokens = [simplemma.lemmatize(t, lang='de') for t in tokens]
    # erst jetzt umwandeln in Kleinbuchstaben
    lemmatized_tokens = [t.lower() for t in lemmatized_tokens]   # neu gegen inaktiv
    text_string = " ".join(lemmatized_tokens)

    # Prüfen, ob genug Wörter nach der Reinigung übrig sind
    if len(lemmatized_tokens) < window_size:
        return f"Fehler: Text hat nur {len(lemmatized_tokens)} Wörter (Fenster: {window_size})."

    # 3. MATTR Berechnung
    lex = LexicalRichness(text_string)
    mattr_score = lex.mattr(window_size=window_size)

    return mattr_score

# Berechnung
CDU_mattr_WP_alles = clean_and_analyze_german(CDU_text)
SPD_mattr_WP_alles = clean_and_analyze_german(SPD_text)
Linke_mattr_WP_alles = clean_and_analyze_german(Linke_text)
AfD_mattr_WP_alles = clean_and_analyze_german(AfD_text)
Grüne_mattr_WP_alles = clean_and_analyze_german(Grüne_text)
FDP_mattr_WP_alles = clean_and_analyze_german(FDP_text)

**2. Teil: Zeichendichte (für Wahlprogramme)**


In [9]:
def analyze_density(text):
    # Alle Wörter/Token zählen (Satzzeichen ausschließen)
    # Findet alle Wörter (inkl. Bindestrichen) und isolierte Zahlen
    raw_words = re.findall(r'\b[\w-]+\b', text)
    total_words = len(raw_words)

    # Zahlen extrahieren (isoliert, inkl. Dezimal- und Tausendertrennzeichen)
    number_pattern = r'^\d+(?:[.,]\d+)*$'                     # ALT         r'\b\d+(?:[.,]\d+)*\b'
    numbers = [t for t in raw_words if re.match(number_pattern, t)]  # ALT   re.findall(number_pattern, text)
    total_numbers = len(numbers)

    # Dichte pro 1000 Wörter
    density_per_1000_words = (total_numbers / total_words * 1000) if total_words > 0 else 0

    return density_per_1000_words

# Berechnung pro 1.000 Wörter
CDU_zahlendichte_WP = analyze_density(CDU_text)
SPD_zahlendichte_WP = analyze_density(SPD_text)
Linke_zahlendichte_WP = analyze_density(Linke_text)
AfD_zahlendichte_WP = analyze_density(AfD_text)
Grüne_zahlendichte_WP = analyze_density(Grüne_text)
FDP_zahlendichte_WP = analyze_density(FDP_text)

**Bundestagsreden**

In [10]:
import json
import urllib.request, urllib.parse, urllib.error
stammdaten = ("MDB_STAMMDATEN.XML")  # mit den Parteizugehörigkeiten
from bs4 import BeautifulSoup
with open(stammdaten, "r", encoding="utf-8") as f:
  	soup2 = BeautifulSoup(f, "xml")

redner_id_mdb = dict()
for mdb in soup2.find_all("MDB"):
	id_tag = mdb.find("ID")
	party_tag = mdb.find("PARTEI_KURZ")
	mdb_id = id_tag.text if id_tag else None
	party = party_tag.text if party_tag else None
	redner_id_mdb[mdb_id] = party

# da wo keine Parteizugehörigkeit aus den Stammdaten ersichtlich, manuelles Hinzufügen diverser Nummern
redner_id_mdb["11002735"]  # MERZ
redner_id_mdb["999990151"] = "SPD"
redner_id_mdb["999990133"] = "SPD"
redner_id_mdb["999990074"] = "SPD"
redner_id_mdb["11005217 999990074"] = "SPD"
redner_id_mdb["999990119"] = "SPD"
redner_id_mdb["999990149"] = "SPD"
redner_id_mdb["999990080"] = "parteilos"
redner_id_mdb["999990142"] = "parteilos"
redner_id_mdb["999990154"] = "parteilos"
redner_id_mdb["999990152"] = "CDU"
redner_id_mdb["999990153"] = "CDU"
redner_id_mdb["999990150"] = "CDU"
redner_id_mdb["999990141"] = "CSU"
redner_id_mdb["999990193"] = "SPD"
redner_id_mdb["999990093"] = "SPD"
redner_id_mdb["999990120"] = "SPD"
redner_id_mdb["999990129"] = "SPD"
redner_id_mdb["999990145"] = "SPD"
redner_id_mdb["999990144"] = "CDU"
redner_id_mdb["999990125"] = "CDU"
redner_id_mdb["999990147"] = "CSU"
redner_id_mdb["999990146"] = "SPD"
redner_id_mdb["999990121"] = "SPD"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990148"] = "SPD"
redner_id_mdb["999990122"] = "BÜNDNIS 90/DIE GRÜNEN"
redner_id_mdb["999990123"] = "DIE LINKE"
redner_id_mdb["999990124"] = "SPD"
redner_id_mdb["999990078"] = "FDP"
redner_id_mdb["999990082"] = "SPD"

In [11]:
# Import der Bundestagsreden via gültigem API-Key für die 5 Betrachtungszeiträume
# API-Key gültig bis zunächst Ende Mai 2027
api = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"

# Eingrenzung Zeitraum Nr. 1, Umwandlung in json-Format & Extraktion der XML-URLs (=Protokolle) aus dem Dictionary
html_zeitraum_1 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2022-01-01&f.datum.end=2022-03-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_1 = json.loads(html_zeitraum_1)
# Erfassen der XML-URLs in einer Liste
data_zeitraum_1_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_1["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 17 Protokolle

# Eingrenzung Zeitraum Nr. 2...
html_zeitraum_2 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2023-10-01&f.datum.end=2023-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_2 = json.loads(html_zeitraum_2)
data_zeitraum_2_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_2["documents"]
    if "xml_url" in doc["fundstelle"]
]  # 19 Protokolle

# Eingrenzung Zeitraum Nr. 3...
html_zeitraum_3 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2024-10-01&f.datum.end=2024-12-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_3 = json.loads(html_zeitraum_3)
data_zeitraum_3_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_3["documents"]
    if "xml_url" in doc["fundstelle"]
]   # 19 Protokolle

# Eingrenzung Zeitraum Nr. 4...
html_zeitraum_4 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-01-01&f.datum.end=2025-02-23&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_4 = json.loads(html_zeitraum_4)
data_zeitraum_4_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_4["documents"]
    if "xml_url" in doc["fundstelle"]
]    # 4 Protokolle

# Eingrenzung Zeitraum Nr. 5...
html_zeitraum_5 = urllib.request.urlopen("https://search.dip.bundestag.de/api/v1/plenarprotokoll?f.datum.start=2025-05-01&f.datum.end=2025-07-31&format=json&apikey=R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ").read()
data_zeitraum_5 = json.loads(html_zeitraum_5)
data_zeitraum_5_X = [
    doc["fundstelle"]["xml_url"]
    for doc in data_zeitraum_5["documents"]
    if "xml_url" in doc["fundstelle"]   
]    # 18 Protokolle

In [12]:
# Anzahl Reden pro Partei und Zeitraum
# zunächst leere Listen
text_liste_CDU_1 = []
text_liste_CDU_2 = []
text_liste_CDU_3 = []
text_liste_CDU_4 = []
text_liste_CDU_5 = []

text_liste_SPD_1 = []
text_liste_SPD_2 = []
text_liste_SPD_3 = []
text_liste_SPD_4 = []
text_liste_SPD_5 = []

text_liste_FDP_1 = []
text_liste_FDP_2 = []
text_liste_FDP_3 = []
text_liste_FDP_4 = []
text_liste_FDP_5 = []

text_liste_Grüne_1 = []
text_liste_Grüne_2 = []
text_liste_Grüne_3 = []
text_liste_Grüne_4 = []
text_liste_Grüne_5 = []

text_liste_Linke_1 = []
text_liste_Linke_2 = []
text_liste_Linke_3 = []
text_liste_Linke_4 = []
text_liste_Linke_5 = []

text_liste_AfD_1 = []
text_liste_AfD_2 = []
text_liste_AfD_3 = []
text_liste_AfD_4 = []
text_liste_AfD_5 = []

# Durchlauf aller extrahierter XML-URLs pro Zeitraum (1 bis 5), Anhängen der Reden pro Partei und Zeitraum an die obigen Listen
for i in data_zeitraum_1_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_1.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_1.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_1.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_1.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_1.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_1.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_1.append(p)

for i in data_zeitraum_2_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_2.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_2.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_2.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_2.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_2.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_2.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_2.append(p)

for i in data_zeitraum_3_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_3.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_3.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_3.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_3.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_3.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_3.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_3.append(p)

for i in data_zeitraum_4_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_4.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_4.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_4.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_4.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_4.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_4.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_4.append(p)

for i in data_zeitraum_5_X:
  html_x = urllib.request.urlopen(i).read()
  soup_x = BeautifulSoup(html_x, "xml")
  for rede in soup_x.find_all("rede"):
    p = rede.find_all("p")
    redner = rede.find("redner")
    id = redner.get("id")
    if redner_id_mdb[id] == "CDU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "CSU":
      text_liste_CDU_5.append(p)
    if redner_id_mdb[id] == "SPD":
      text_liste_SPD_5.append(p)
    if redner_id_mdb[id] == "DIE LINKE.":
      text_liste_Linke_5.append(p)
    if redner_id_mdb[id] == "AfD":
      text_liste_AfD_5.append(p)
    if redner_id_mdb[id] == "FDP":
      text_liste_FDP_5.append(p)
    if redner_id_mdb[id] == "BÜNDNIS 90/DIE GRÜNEN":
      text_liste_Grüne_5.append(p)
    if redner_id_mdb[id] == "GRÜNE":
      text_liste_Grüne_5.append(p)

In [13]:
# Bereinigung der Texte um Texte der Bundestagspräsidentin und sonstigen Nicht-Rede-Elementen

def bereinige_text_liste(liste):
    bereinigte_liste = []
    for i in liste:
        i = str(i[1:]).replace('<p klasse="J_1">', "")
        i = i.replace('<p klasse="J">', "")
        i = i.replace('<p klasse="O">', "")
        i = i.replace("<p klasse=", "")
        i = re.sub(r'-\s+', '', i)    # neu
        i = re.sub(r'[^\w\s\.!\?€,,:\(\)\-%]', '', i)  # neu
        i = i.replace("p, ", " ")
        i = i.replace(".p, p", ".")
        i = i.replace(":p, ",": ")
        i = re.sub(r'Die\s+nächste\s+Rednerin\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'Der\s+nächste\s+Redner\s+ist\s+[\w\s.-]+\s+für\s+die\s+[\w\s.-]+[\.!?]', '', i)
        i = re.sub(r'[D|d]er nächste Redner in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'[D|d]ie nächste Rednerin in der Debatte ist [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)?\.?', '', i)
        i = re.sub(r'rednerredner\s+[\w\s. -ÄÖÜäöüß]+:', '', i)
        i = re.sub(r'[F|f]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+-Fraktion erhält das Wort [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'Als\s+Nächstes\s+spricht\s+[\w\s. -ÄÖÜäöüß]+[\.!?]', '', i)
        i = re.sub(r'[I|i]ch darf für die Fraktion (?:[A-Zßäöüa-z\s\/äöü]+) [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? [A-Z][a-zßäöü]+(?:-[A-Z][a-zßäöü]+)? aufrufen\.?', '', i)
        i = re.sub(r'[I|i]ch erteile das Wort für die nächste Rede (?:[A-ZßäöüÄÖÜa-z\s\/äöü]+) [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?)?\.?', '', i)
        i = re.sub(r'[I|i]ch erteile als Nächstes das Wort de[mr] Abgeordneten [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\/]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)* für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[I|i]ch darf aufrufen für die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[D|d]ie nächste Rede hält [A-Z][a-zßäöüÄÖÜ]+(?: [A-Z][a-zßäöüÄÖÜ]+)*(?:\.)?', '', i)
        i = re.sub(r'[A[Aa]ls nächste(?:r)? (?:Rednerin|Redner) hat [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'[E[Ee]benfalls zur ersten Rede erteile ich das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort erteilen\.?', '', i)
        i = re.sub(r'[Ii]ch darf [A-ZßäöüÄÖÜa-z\s\-\/\.]+ aufrufen\.?', '', i)
        i = re.sub(r'[Ii]ch erteile das Wort [A-ZßäöüÄÖÜa-z\s\-\/\.]+\.?', '', i)
        i = re.sub(r'[Vv]ielen Dank und Gratulation zu Ihrer ersten Rede, (?:Frau|Herr) [A-ZßäöüÄÖÜa-z\s\-\/\.]+(?:\.)?', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu seiner ersten Rede\.', '', i)
        i = re.sub(r'[Ff]ür die [A-Za-zßäöüÄÖÜ\s\-\/0-9]+ das Wort zu ihrer ersten Rede\.', '', i)
        i = re.sub(r'[Dd]er nächste Redner in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ie nächste Rednerin in der Debatte: für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Dd]ann rufe ich [A-ZßäöüÄÖÜa-z\s\-\/\.]+ in der Debatte auf: [A-Z][a-zßäöüÄÖÜ]+(?:-[A-Z][a-zßäöüÄÖÜ]+)?(?: [A-Z][a-zßäöüÄÖÜ]+)? für [A-Za-zßäöüÄÖÜ\s\-\/0-9]+(?:\.)?', '', i)
        i = re.sub(r'[Zz]u (?:seiner|ihrer) ersten Rede hat nun [A-ZßäöüÄÖÜa-z\s\-\/\.]+ das Wort\.?', '', i)
        i = re.sub(r'von der [A-Za-zßäöüÄÖÜ\s\-\/0-9]+, für (?:den|die) es hier die erste Rede ist\.?', '', i)
        i = i.replace("Vielen Dank Ihnen. ", "")
        i = i.removesuffix('p')
        i = re.sub(r"\s+", " ", i)
        i = i.replace("\xa0", "")
        i = i.replace("</p>", "")
        i = i.replace("</p", "")
        i = i.replace("p, p, ", "")
        i = i.lstrip()
        i = i.rstrip(" ")
        bereinigte_liste.append(i)
    return bereinigte_liste

# Zeitraum 1
text_liste_CDU_1_clean = bereinige_text_liste(text_liste_CDU_1)
text_liste_SPD_1_clean = bereinige_text_liste(text_liste_SPD_1)
text_liste_Linke_1_clean = bereinige_text_liste(text_liste_Linke_1)
text_liste_AfD_1_clean = bereinige_text_liste(text_liste_AfD_1)
text_liste_Grüne_1_clean = bereinige_text_liste(text_liste_Grüne_1)
text_liste_FDP_1_clean = bereinige_text_liste(text_liste_FDP_1)

# Zeitraum 2
text_liste_CDU_2_clean = bereinige_text_liste(text_liste_CDU_2)
text_liste_SPD_2_clean = bereinige_text_liste(text_liste_SPD_2)
text_liste_Linke_2_clean = bereinige_text_liste(text_liste_Linke_2)
text_liste_AfD_2_clean = bereinige_text_liste(text_liste_AfD_2)
text_liste_Grüne_2_clean = bereinige_text_liste(text_liste_Grüne_2)
text_liste_FDP_2_clean = bereinige_text_liste(text_liste_FDP_2)

# Zeitraum 3
text_liste_CDU_3_clean = bereinige_text_liste(text_liste_CDU_3)
text_liste_SPD_3_clean = bereinige_text_liste(text_liste_SPD_3)
text_liste_Linke_3_clean = bereinige_text_liste(text_liste_Linke_3)
text_liste_AfD_3_clean = bereinige_text_liste(text_liste_AfD_3)
text_liste_Grüne_3_clean = bereinige_text_liste(text_liste_Grüne_3)
text_liste_FDP_3_clean = bereinige_text_liste(text_liste_FDP_3)

# Zeitraum 4
text_liste_CDU_4_clean = bereinige_text_liste(text_liste_CDU_4)
text_liste_SPD_4_clean = bereinige_text_liste(text_liste_SPD_4)
text_liste_Linke_4_clean = bereinige_text_liste(text_liste_Linke_4)
text_liste_AfD_4_clean = bereinige_text_liste(text_liste_AfD_4)
text_liste_Grüne_4_clean = bereinige_text_liste(text_liste_Grüne_4)
text_liste_FDP_4_clean = bereinige_text_liste(text_liste_FDP_4)

# Zeitraum 5
text_liste_CDU_5_clean = bereinige_text_liste(text_liste_CDU_5)
text_liste_SPD_5_clean = bereinige_text_liste(text_liste_SPD_5)
text_liste_Linke_5_clean = bereinige_text_liste(text_liste_Linke_5)
text_liste_AfD_5_clean = bereinige_text_liste(text_liste_AfD_5)
text_liste_Grüne_5_clean = bereinige_text_liste(text_liste_Grüne_5)
text_liste_FDP_5_clean = bereinige_text_liste(text_liste_FDP_5)

In [14]:
# aus jeder Liste einen großen String pro Partei und Betrachtungszeitraum
text_liste_CDU_1_str = "".join(text_liste_CDU_1_clean)
text_liste_SPD_1_str = "".join(text_liste_SPD_1_clean)
text_liste_AfD_1_str = "".join(text_liste_AfD_1_clean)
text_liste_Linke_1_str = "".join(text_liste_Linke_1_clean)
text_liste_Grüne_1_str = "".join(text_liste_Grüne_1_clean)
text_liste_FDP_1_str = "".join(text_liste_FDP_1_clean)

text_liste_CDU_2_str = "".join(text_liste_CDU_2_clean)
text_liste_SPD_2_str = "".join(text_liste_SPD_2_clean)
text_liste_AfD_2_str = "".join(text_liste_AfD_2_clean)
text_liste_Linke_2_str = "".join(text_liste_Linke_2_clean)
text_liste_Grüne_2_str = "".join(text_liste_Grüne_2_clean)
text_liste_FDP_2_str = "".join(text_liste_FDP_2_clean)

text_liste_CDU_3_str = "".join(text_liste_CDU_3_clean)
text_liste_SPD_3_str = "".join(text_liste_SPD_3_clean)
text_liste_AfD_3_str = "".join(text_liste_AfD_3_clean)
text_liste_Linke_3_str = "".join(text_liste_Linke_3_clean)
text_liste_Grüne_3_str = "".join(text_liste_Grüne_3_clean)
text_liste_FDP_3_str = "".join(text_liste_FDP_3_clean)

text_liste_CDU_4_str = "".join(text_liste_CDU_4_clean)
text_liste_SPD_4_str = "".join(text_liste_SPD_4_clean)
text_liste_AfD_4_str = "".join(text_liste_AfD_4_clean)
text_liste_Linke_4_str = "".join(text_liste_Linke_4_clean)
text_liste_Grüne_4_str = "".join(text_liste_Grüne_4_clean)
text_liste_FDP_4_str = "".join(text_liste_FDP_4_clean)

text_liste_CDU_5_str = "".join(text_liste_CDU_5_clean)
text_liste_SPD_5_str = "".join(text_liste_SPD_5_clean)
text_liste_AfD_5_str = "".join(text_liste_AfD_5_clean)
text_liste_Linke_5_str = "".join(text_liste_Linke_5_clean)
text_liste_Grüne_5_str = "".join(text_liste_Grüne_5_clean)
text_liste_FDP_5_str = "".join(text_liste_FDP_5_clean)

**1. Teil: lexical richness (für Bundestagsreden)**

In [15]:
# Bereinigen der Texte => Umformen von Abkürzungen in Text um Eindruck von Satzende zu vermeiden

# def bereinige_abkuerzungen(text):    -> s. oben

text_liste_CDU_1_str = bereinige_abkuerzungen(text_liste_CDU_1_str)
text_liste_SPD_1_str = bereinige_abkuerzungen(text_liste_SPD_1_str)
text_liste_AfD_1_str = bereinige_abkuerzungen(text_liste_AfD_1_str)
text_liste_Linke_1_str = bereinige_abkuerzungen(text_liste_Linke_1_str)
text_liste_Grüne_1_str = bereinige_abkuerzungen(text_liste_Grüne_1_str)
text_liste_FDP_1_str = bereinige_abkuerzungen(text_liste_FDP_1_str)

text_liste_CDU_2_str = bereinige_abkuerzungen(text_liste_CDU_2_str)
text_liste_SPD_2_str = bereinige_abkuerzungen(text_liste_SPD_2_str)
text_liste_AfD_2_str = bereinige_abkuerzungen(text_liste_AfD_2_str)
text_liste_Linke_2_str = bereinige_abkuerzungen(text_liste_Linke_2_str)
text_liste_Grüne_2_str = bereinige_abkuerzungen(text_liste_Grüne_2_str)
text_liste_FDP_2_str = bereinige_abkuerzungen(text_liste_FDP_2_str)

text_liste_CDU_3_str = bereinige_abkuerzungen(text_liste_CDU_3_str)
text_liste_SPD_3_str = bereinige_abkuerzungen(text_liste_SPD_3_str)
text_liste_AfD_3_str = bereinige_abkuerzungen(text_liste_AfD_3_str)
text_liste_Linke_3_str = bereinige_abkuerzungen(text_liste_Linke_3_str)
text_liste_Grüne_3_str = bereinige_abkuerzungen(text_liste_Grüne_3_str)
text_liste_FDP_3_str = bereinige_abkuerzungen(text_liste_FDP_3_str)

text_liste_CDU_4_str = bereinige_abkuerzungen(text_liste_CDU_4_str)
text_liste_SPD_4_str = bereinige_abkuerzungen(text_liste_SPD_4_str)
text_liste_AfD_4_str = bereinige_abkuerzungen(text_liste_AfD_4_str)
text_liste_Linke_4_str = bereinige_abkuerzungen(text_liste_Linke_4_str)
text_liste_Grüne_4_str = bereinige_abkuerzungen(text_liste_Grüne_4_str)
text_liste_FDP_4_str = bereinige_abkuerzungen(text_liste_FDP_4_str)

text_liste_CDU_5_str = bereinige_abkuerzungen(text_liste_CDU_5_str)
text_liste_SPD_5_str = bereinige_abkuerzungen(text_liste_SPD_5_str)
text_liste_AfD_5_str = bereinige_abkuerzungen(text_liste_AfD_5_str)
text_liste_Linke_5_str = bereinige_abkuerzungen(text_liste_Linke_5_str)
text_liste_Grüne_5_str = bereinige_abkuerzungen(text_liste_Grüne_5_str)
text_liste_FDP_5_str = bereinige_abkuerzungen(text_liste_FDP_5_str)

In [16]:
# ohne stopwords

# automatisierte Funktion  QUELLE: Gemini
def clean_and_analyze_german(text, window_size=100):
    text_cleaned = re.sub(r'[^a-zA-ZäöüÄÖÜß\s]', '', text)
    text_cleaned = text_cleaned.lower()

    # 2. Tokenisierung
    # und zusätzlich leere Strings rausfiltern, die durch re.sub entstehen könnten
    tokens = re.findall(r'\b\w+\b', text_cleaned)
    tokens = [t for t in tokens if t.strip()]

    # Bereinigen um stopwords
    tokens_clear = [word for word in tokens if word not in stopwords]

    # Lemmatisierung (behält die originale Wortanzahl und Reihenfolge bei)
    lemmatized_tokens = [simplemma.lemmatize(t, lang='de') for t in tokens_clear]
    text_string = " ".join(lemmatized_tokens)

    # Prüfen, ob genug Wörter nach der Reinigung übrig sind
    if len(lemmatized_tokens) < window_size:
        return f"Fehler: Text hat nur {len(lemmatized_tokens)} Wörter (Fenster: {window_size})."

    # 3. MATTR Berechnung
    lex = LexicalRichness(text_string)
    mattr_score = lex.mattr(window_size=window_size)

    return mattr_score

# Berechnung
CDU_mattr_BT1 = clean_and_analyze_german(text_liste_CDU_1_str)
CDU_mattr_BT2 = clean_and_analyze_german(text_liste_CDU_2_str)
CDU_mattr_BT3 = clean_and_analyze_german(text_liste_CDU_3_str)
CDU_mattr_BT4 = clean_and_analyze_german(text_liste_CDU_4_str)
CDU_mattr_BT5 = clean_and_analyze_german(text_liste_CDU_5_str)

SPD_mattr_BT1 = clean_and_analyze_german(text_liste_SPD_1_str)
SPD_mattr_BT2 = clean_and_analyze_german(text_liste_SPD_2_str)
SPD_mattr_BT3 = clean_and_analyze_german(text_liste_SPD_3_str)
SPD_mattr_BT4 = clean_and_analyze_german(text_liste_SPD_4_str)
SPD_mattr_BT5 = clean_and_analyze_german(text_liste_SPD_5_str)

Linke_mattr_BT1 = clean_and_analyze_german(text_liste_Linke_1_str)
Linke_mattr_BT2 = clean_and_analyze_german(text_liste_Linke_2_str)
Linke_mattr_BT3 = clean_and_analyze_german(text_liste_Linke_3_str)
Linke_mattr_BT4 = clean_and_analyze_german(text_liste_Linke_4_str)
Linke_mattr_BT5 = clean_and_analyze_german(text_liste_Linke_5_str)

Grüne_mattr_BT1 = clean_and_analyze_german(text_liste_Grüne_1_str)
Grüne_mattr_BT2 = clean_and_analyze_german(text_liste_Grüne_2_str)
Grüne_mattr_BT3 = clean_and_analyze_german(text_liste_Grüne_3_str)
Grüne_mattr_BT4 = clean_and_analyze_german(text_liste_Grüne_4_str)
Grüne_mattr_BT5 = clean_and_analyze_german(text_liste_Grüne_5_str)

AfD_mattr_BT1 = clean_and_analyze_german(text_liste_AfD_1_str)
AfD_mattr_BT2 = clean_and_analyze_german(text_liste_AfD_2_str)
AfD_mattr_BT3 = clean_and_analyze_german(text_liste_AfD_3_str)
AfD_mattr_BT4 = clean_and_analyze_german(text_liste_AfD_4_str)
AfD_mattr_BT5 = clean_and_analyze_german(text_liste_AfD_5_str)

FDP_mattr_BT1 = clean_and_analyze_german(text_liste_FDP_1_str)
FDP_mattr_BT2 = clean_and_analyze_german(text_liste_FDP_2_str)
FDP_mattr_BT3 = clean_and_analyze_german(text_liste_FDP_3_str)
FDP_mattr_BT4 = clean_and_analyze_german(text_liste_FDP_4_str)

In [17]:
# mit allen Wörtern, d.h. auch stopwords

# automatisierte Funktion  QUELLE: Gemini
def clean_and_analyze_german(text, window_size=100):
    text_cleaned = re.sub(r'[^a-zA-ZäöüÄÖÜß\s]', '', text)
    text_cleaned = text_cleaned.lower()

    # 2. Tokenisierung
    # und zusätzlich leere Strings rausfiltern, die durch re.sub entstehen könnten
    tokens = re.findall(r'\b\w+\b', text_cleaned)
    tokens = [t for t in tokens if t.strip()]

    # Lemmatisierung (behält die originale Wortanzahl und Reihenfolge bei)
    lemmatized_tokens = [simplemma.lemmatize(t, lang='de') for t in tokens]
    text_string = " ".join(lemmatized_tokens)

    # Prüfen, ob genug Wörter nach der Reinigung übrig sind
    if len(lemmatized_tokens) < window_size:
        return f"Fehler: Text hat nur {len(lemmatized_tokens)} Wörter (Fenster: {window_size})."

    # 3. MATTR Berechnung
    lex = LexicalRichness(text_string)
    mattr_score = lex.mattr(window_size=window_size)

    return mattr_score

# Berechnung
CDU_mattr_BT1_alles = clean_and_analyze_german(text_liste_CDU_1_str)
CDU_mattr_BT2_alles = clean_and_analyze_german(text_liste_CDU_2_str)
CDU_mattr_BT3_alles = clean_and_analyze_german(text_liste_CDU_3_str)
CDU_mattr_BT4_alles = clean_and_analyze_german(text_liste_CDU_4_str)
CDU_mattr_BT5_alles = clean_and_analyze_german(text_liste_CDU_5_str)

SPD_mattr_BT1_alles = clean_and_analyze_german(text_liste_SPD_1_str)
SPD_mattr_BT2_alles = clean_and_analyze_german(text_liste_SPD_2_str)
SPD_mattr_BT3_alles = clean_and_analyze_german(text_liste_SPD_3_str)
SPD_mattr_BT4_alles = clean_and_analyze_german(text_liste_SPD_4_str)
SPD_mattr_BT5_alles = clean_and_analyze_german(text_liste_SPD_5_str)

Linke_mattr_BT1_alles = clean_and_analyze_german(text_liste_Linke_1_str)
Linke_mattr_BT2_alles = clean_and_analyze_german(text_liste_Linke_2_str)
Linke_mattr_BT3_alles = clean_and_analyze_german(text_liste_Linke_3_str)
Linke_mattr_BT4_alles = clean_and_analyze_german(text_liste_Linke_4_str)
Linke_mattr_BT5_alles = clean_and_analyze_german(text_liste_Linke_5_str)

Grüne_mattr_BT1_alles = clean_and_analyze_german(text_liste_Grüne_1_str)
Grüne_mattr_BT2_alles = clean_and_analyze_german(text_liste_Grüne_2_str)
Grüne_mattr_BT3_alles = clean_and_analyze_german(text_liste_Grüne_3_str)
Grüne_mattr_BT4_alles = clean_and_analyze_german(text_liste_Grüne_4_str)
Grüne_mattr_BT5_alles = clean_and_analyze_german(text_liste_Grüne_5_str)

AfD_mattr_BT1_alles = clean_and_analyze_german(text_liste_AfD_1_str)
AfD_mattr_BT2_alles = clean_and_analyze_german(text_liste_AfD_2_str)
AfD_mattr_BT3_alles = clean_and_analyze_german(text_liste_AfD_3_str)
AfD_mattr_BT4_alles = clean_and_analyze_german(text_liste_AfD_4_str)
AfD_mattr_BT5_alles = clean_and_analyze_german(text_liste_AfD_5_str)

FDP_mattr_BT1_alles = clean_and_analyze_german(text_liste_FDP_1_str)
FDP_mattr_BT2_alles = clean_and_analyze_german(text_liste_FDP_2_str)
FDP_mattr_BT3_alles = clean_and_analyze_german(text_liste_FDP_3_str)
FDP_mattr_BT4_alles = clean_and_analyze_german(text_liste_FDP_4_str)

**Grafik mit MATTR-Werten auf Wahlprogrammen und für Bundestagsreden, 1x mit allen Wörtern und 1x Stopp-Wörter weggelassen**

In [18]:
# PandaFrame-Erstellung
df_mattr = pd.DataFrame({
    ("Wahlprogramm", "no_sw"): [CDU_mattr_WP, SPD_mattr_WP, Linke_mattr_WP, AfD_mattr_WP, Grüne_mattr_WP, FDP_mattr_WP],
    ("Wahlprogramm", "all"): [CDU_mattr_WP_alles, SPD_mattr_WP_alles, Linke_mattr_WP_alles, AfD_mattr_WP_alles, Grüne_mattr_WP_alles, FDP_mattr_WP_alles],
    ("Periode 1", "no_sw"): [CDU_mattr_BT1, SPD_mattr_BT1, Linke_mattr_BT1, AfD_mattr_BT1, Grüne_mattr_BT1, FDP_mattr_BT1],
    ("Periode 1", "all"): [CDU_mattr_BT1_alles, SPD_mattr_BT1_alles, Linke_mattr_BT1_alles, AfD_mattr_BT1_alles, Grüne_mattr_BT1_alles, FDP_mattr_BT1_alles],
    ("Periode 2", "no_sw"): [CDU_mattr_BT2, SPD_mattr_BT2, Linke_mattr_BT2, AfD_mattr_BT2, Grüne_mattr_BT2, FDP_mattr_BT2],
    ("Periode 2", "all"): [CDU_mattr_BT2_alles, SPD_mattr_BT2_alles, Linke_mattr_BT2_alles, AfD_mattr_BT2_alles, Grüne_mattr_BT2_alles, FDP_mattr_BT2_alles],
    ("Periode 3", "no_sw"): [CDU_mattr_BT3, SPD_mattr_BT3, Linke_mattr_BT3, AfD_mattr_BT3, Grüne_mattr_BT3, FDP_mattr_BT3],
    ("Periode 3", "all"): [CDU_mattr_BT3_alles, SPD_mattr_BT3_alles, Linke_mattr_BT3_alles, AfD_mattr_BT3_alles, Grüne_mattr_BT3_alles, FDP_mattr_BT3_alles],
    ("Periode 4", "no_sw"): [CDU_mattr_BT4, SPD_mattr_BT4, Linke_mattr_BT4, AfD_mattr_BT4, Grüne_mattr_BT4, FDP_mattr_BT4],
    ("Periode 4", "all"): [CDU_mattr_BT4_alles, SPD_mattr_BT4_alles, Linke_mattr_BT4_alles, AfD_mattr_BT4_alles, Grüne_mattr_BT4_alles, FDP_mattr_BT4_alles],
    ("Periode 5", "no_sw"): [CDU_mattr_BT5, SPD_mattr_BT5, Linke_mattr_BT5, AfD_mattr_BT5, Grüne_mattr_BT5,0.0],
    ("Periode 5", "all"): [CDU_mattr_BT5_alles, SPD_mattr_BT5_alles, Linke_mattr_BT5_alles, AfD_mattr_BT5_alles, Grüne_mattr_BT5_alles,0.0],
}).round(2)

df_mattr.index = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_mattr = df_mattr.replace(0.0, "-")
df_mattr

Wahlprogramm       Periode 1       Periode 2       Periode 3        \
               no_sw   all     no_sw   all     no_sw   all     no_sw   all   
CDU/CSU         0.85  0.71      0.84  0.69      0.85  0.69      0.84  0.69   
SPD             0.86  0.71      0.85  0.69      0.85  0.69      0.85  0.69   
Linke           0.86  0.71      0.86  0.71      0.88  0.71      0.87  0.71   
AfD             0.88  0.73      0.86  0.71      0.87  0.71      0.87  0.71   
Grüne           0.88  0.71      0.85  0.69      0.85  0.69      0.85  0.69   
FDP             0.87  0.72      0.85  0.69      0.85  0.69      0.84  0.69   

        Periode 4       Periode 5        
            no_sw   all     no_sw   all  
CDU/CSU      0.84  0.68      0.85  0.69  
SPD          0.85  0.68      0.85  0.69  
Linke        0.87  0.71      0.87  0.71  
AfD          0.86  0.70      0.88  0.72  
Grüne        0.84  0.68      0.85  0.69  
FDP          0.85  0.69         -     -

In [19]:
# in LaTex überführen

latex_code = df_mattr.to_latex(index=False, caption='Reden', label='tab:meine_tabelle',column_format='cc||cc||cc||cc||cc||cc')
with open('tabelle_mattr.tex', 'w') as f:
    f.write(latex_code)

**2. Teil: Zeichendichte (für Bundestagsreden)**

In [20]:
#   def analyze_density(text):   --> Formel s. o.

# Berechnung pro 1.000 Wörter
CDU_zahlendichte_BT1 = analyze_density(text_liste_CDU_1_str)
CDU_zahlendichte_BT2 = analyze_density(text_liste_CDU_2_str)
CDU_zahlendichte_BT3 = analyze_density(text_liste_CDU_3_str)
CDU_zahlendichte_BT4 = analyze_density(text_liste_CDU_4_str)
CDU_zahlendichte_BT5 = analyze_density(text_liste_CDU_5_str)

SPD_zahlendichte_BT1 = analyze_density(text_liste_SPD_1_str)
SPD_zahlendichte_BT2 = analyze_density(text_liste_SPD_2_str)
SPD_zahlendichte_BT3 = analyze_density(text_liste_SPD_3_str)
SPD_zahlendichte_BT4 = analyze_density(text_liste_SPD_4_str)
SPD_zahlendichte_BT5 = analyze_density(text_liste_SPD_5_str)

Linke_zahlendichte_BT1 = analyze_density(text_liste_Linke_1_str)
Linke_zahlendichte_BT2 = analyze_density(text_liste_Linke_2_str)
Linke_zahlendichte_BT3 = analyze_density(text_liste_Linke_3_str)
Linke_zahlendichte_BT4 = analyze_density(text_liste_Linke_4_str)
Linke_zahlendichte_BT5 = analyze_density(text_liste_Linke_5_str)

Grüne_zahlendichte_BT1 = analyze_density(text_liste_Grüne_1_str)
Grüne_zahlendichte_BT2 = analyze_density(text_liste_Grüne_2_str)
Grüne_zahlendichte_BT3 = analyze_density(text_liste_Grüne_3_str)
Grüne_zahlendichte_BT4 = analyze_density(text_liste_Grüne_4_str)
Grüne_zahlendichte_BT5 = analyze_density(text_liste_Grüne_5_str)

AfD_zahlendichte_BT1 = analyze_density(text_liste_AfD_1_str)
AfD_zahlendichte_BT2 = analyze_density(text_liste_AfD_2_str)
AfD_zahlendichte_BT3 = analyze_density(text_liste_AfD_3_str)
AfD_zahlendichte_BT4 = analyze_density(text_liste_AfD_4_str)
AfD_zahlendichte_BT5 = analyze_density(text_liste_AfD_5_str)

FDP_zahlendichte_BT1 = analyze_density(text_liste_FDP_1_str)
FDP_zahlendichte_BT2 = analyze_density(text_liste_FDP_2_str)
FDP_zahlendichte_BT3 = analyze_density(text_liste_FDP_3_str)
FDP_zahlendichte_BT4 = analyze_density(text_liste_FDP_4_str)

In [21]:
# PandaFrame-Erstellung
df_zahlendichte = pd.DataFrame({
    ("Programm", "num_d"): [CDU_zahlendichte_WP, SPD_zahlendichte_WP, Linke_zahlendichte_WP, AfD_zahlendichte_WP, Grüne_zahlendichte_WP, FDP_zahlendichte_WP],
    ("Periode 1", "num_d"): [CDU_zahlendichte_BT1, SPD_zahlendichte_BT1, Linke_zahlendichte_BT1, AfD_zahlendichte_BT1, Grüne_zahlendichte_BT1, FDP_zahlendichte_BT1],
    ("Periode 2", "num_d"): [CDU_zahlendichte_BT2, SPD_zahlendichte_BT2, Linke_zahlendichte_BT2, AfD_zahlendichte_BT2, Grüne_zahlendichte_BT2, FDP_zahlendichte_BT2],
    ("Periode 3", "num_d"): [CDU_zahlendichte_BT3, SPD_zahlendichte_BT3, Linke_zahlendichte_BT3, AfD_zahlendichte_BT3, Grüne_zahlendichte_BT3, FDP_zahlendichte_BT3],
    ("Periode 4", "num_d"): [CDU_zahlendichte_BT4, SPD_zahlendichte_BT4, Linke_zahlendichte_BT4, AfD_zahlendichte_BT4, Grüne_zahlendichte_BT4, FDP_zahlendichte_BT4],
    ("Periode 5", "num_d"): [CDU_zahlendichte_BT5, SPD_zahlendichte_BT5, Linke_zahlendichte_BT5, AfD_zahlendichte_BT5, Grüne_zahlendichte_BT5,0.0],
}).round(2)

df_zahlendichte.index = ["CDU/CSU", "SPD", "Linke", "AfD", "Grüne", "FDP"]
df_zahlendichte = df_zahlendichte.replace(0.0, "-")
df_zahlendichte

,Programm,Periode 1,Periode 2,Periode 3,Periode 4,Periode 5
,num_d,num_d,num_d,num_d,num_d,num_d
CDU/CSU,2.32,7.99,8.10,7.91,8.23,7.37
SPD,2.82,6.89,6.71,6.23,6.11,7.49
Linke,10.53,10.79,10.34,9.80,9.32,9.77
AfD,6.79,11.30,12.27,10.42,8.26,12.19
Grüne,2.54,6.16,5.80,6.66,6.38,7.28
FDP,2.89,6.78,6.04,6.46,5.81,-


In [22]:
# in LaTex überführen

latex_code = df_zahlendichte.to_latex(index=False, caption='Reden', label='tab:meine_tabelle')
with open('tabelle_zahlendichte.tex', 'w') as f:
    f.write(latex_code)